# 1_Preprocessing_for_QuickNII

This notebook downsample whole brain images acquired from Nikon confocal microscopy. Images were acquired using 4X, 1024, XY size = 4419 x 4419 um, pixel size = 4.16 um. The notebook project all channels together and convert to greyscale images, and rename each image accordingly. 

## How to use:
1. Specify the correct directory
2. Specify the experiment name and animal ID
3. Enter the pixel size of your images
4. Adjust the scale factor based on the pixel size so that the output image is approximately 1500 × 1000 pixels


This notebook was adapted from code originally written by Igor from Neuroinformatics Unit. The original code is provided at the bottom of this notebook.

In [ ]:
from pathlib import Path

import numpy as np
import tifffile
from scipy.ndimage import zoom
from PIL import Image


# Folders

input_dir = Path("/Volumes/Extreme SSD/PyNutil_registration_test/converted_tiff") # <-------------- Change to correct directory
output_dir = Path("/Volumes/Extreme SSD/PyNutil_registration_test/quicknii_png")  # <-------------- Change to correct directory
output_dir.mkdir(exist_ok=True)



# Settings

# specify the name and animal ID
output_prefix = "RELN_COLBOS_1125716"                                               # <-------------- Change to correct name and animal ID

# pixel size of the image
size = 4.316                                                                        # <-------------- Change to correct pixel size (in microns) for your images

# scale_factor downscale the size of the original images, adjust this value to ensure the final image area is approximately 1500x1000 pixels
scale_factor = size / 10  



# Helper functions

# sum depth (Z) and channels (C), leave Y, X dimensions untouched
# this project all the channels and z-stack onto a single plane
def sum_project_all_channels(image):
    """
    Sum all dimensions except the final Y, X dimensions.

    Example:
    Z, C, Y, X -> Y, X
    C, Z, Y, X -> Y, X
    Y, X -> Y, X
    """
    print(f"Original image shape: {image.shape}")

    image = image.astype(np.float32)

    if image.ndim == 2:
        projection = image
    else:
        axes_to_sum = tuple(range(image.ndim - 2))
        projection = image.sum(axis=axes_to_sum)

    print(f"Sum projection shape: {projection.shape}")

    return projection


def convert_to_8bit(image):
    """
    Convert image to 8-bit

    This uses fixed scaling:
    - If values are already 0-255, keep them.
    - If values are 16-bit-like, divide by 256.
    - Values above 255 after conversion are clipped.
    """

    image = image.astype(np.float32)

    max_value = image.max()
    print(f"Maximum intensity before 8-bit conversion: {max_value}")

    # If image looks like 16-bit data, convert to 8-bit using fixed 16-bit to 8-bit scaling
    if max_value > 255:
        image = image / 256.0

    # Clip only to fit PNG 8-bit range
    image = np.clip(image, 0, 255)

    return image.astype(np.uint8)


# Automatically find TIFF files

tiff_files = sorted(
    list(input_dir.glob("*.tif")) +
    list(input_dir.glob("*.tiff"))
)

print(f"Found {len(tiff_files)} TIFF files.")


# Main loop
for idx, tiff_path in enumerate(tiff_files):
    print(f"Processing: {tiff_path.name}")

    im = tifffile.imread(tiff_path)

    projection = sum_project_all_channels(im)

    scaled = zoom(projection, scale_factor)
    print(f"Scaled shape: {scaled.shape}")

    scaled_8bit = convert_to_8bit(scaled)

    scaled_image = Image.fromarray(scaled_8bit).convert("RGB")

    # Use enumerate index for section numbering
    output_name = f"{output_prefix}_s{idx+1:03d}.png"
    output_path = output_dir / output_name

    scaled_image.save(output_path)
    print(f"Saved: {output_path}")
  

print("\nAll done.")

Found 2 TIFF files.
Processing: RELNCOLBOS_1125716_NeuN-488_Reelin-568_13042026_s1_wholebrain-Stitched.tif
Original image shape: (25, 3, 2765, 2765)
Sum projection shape: (2765, 2765)
Scaled shape: (1193, 1193)
Maximum intensity before 8-bit conversion: 117585.6640625
Saved: /Volumes/Extreme SSD/PyNutil_registration_test/registration_images/RELN_COLBOS_1125716_s001.png
Processing: RELNCOLBOS_1125716_NeuN-488_Reelin-568_13042026_s2_wholebrain-Stitched.tif
Original image shape: (25, 3, 2765, 3635)
Sum projection shape: (2765, 3635)
Scaled shape: (1193, 1569)
Maximum intensity before 8-bit conversion: 122196.5078125
Saved: /Volumes/Extreme SSD/PyNutil_registration_test/registration_images/RELN_COLBOS_1125716_s002.png

All done.


Original code developed by Igor

In [ ]:
import tifffile
from scipy.ndimage import zoom
from pathlib import Path

from PIL import Image

file_names = [
    ("RELNCOLBOS_1125841_NeuN-488_Reelin-568_13042026_s1_wholebrain-Stitched.tif", 4.316),
    ("RELNCOLBOS_1125841_NeuN-488_Reelin-568_13042026_s2_wholebrain-Stitched.tif", 4.316),
    ("RELNCOLBOS_1125841_NeuN-488_Reelin-568_13042026_s3_wholebrain-Stitched.tif", 4.316),
    ("RELNCOLBOS_1125841_NeuN-488_Reelin-568_13042026_s4_wholebrain-Stitched.tif", 4.316),
    ("RELNCOLBOS_1125841_NeuN-488_Reelin-568_13042026_s5_wholebrain-Stitched.tif", 4.316),
    ("RELNCOLBOS_1125841_NeuN-488_Reelin-568_13042026_s6_wholebrain-Stitched.tif", 4.316),
]

for idx, (file_name, size) in enumerate(file_names):
    path = Path.home() / file_name
    im = tifffile.imread(path)
    print(f"Original shape of {file_name}: {im.shape}")
    max_proj = im.sum(axis=(0, 1)).astype("uint16")
    print(f"Max projection shape of {file_name}: {max_proj.shape}")
    scaled = zoom(max_proj, size/25)
    print(f"Scaled shape of {file_name}: {scaled.shape}")
    scaled_image = Image.fromarray(scaled)
    new_name = file_name.replace(".tif", f"_s{idx+1:04d}.png")
    scaled_image.save(new_name)